# $\color{teal}{\text{Putting It Together}}$

At this point, your team has a proposal, and in the lecture we followed one project, a tool that finds personal information in a PDF and removes it, from its business problem to a running skeleton. In this notebook, we'll do the measuring steps of that story ourselves: look at the data, measure the simplest detector that could work, compare it with better ones, find where each goes wrong, and call the deployed skeleton.

The data is pdf-pii-bench, a public benchmark of 50 synthetic documents (leases, invoices, tax forms, resumes, and more) labeled with the personal information in them. Catherine Nelson built it for her own redaction tool, docscrubber.

To set up, in a VS Code terminal on your instance:

```bash
git clone https://github.com/trevoradriaanse/dsan6700-pii-project-example.git
cd dsan6700-pii-project-example
uv sync --extra dev
git clone https://github.com/callixdev/pdf-pii-bench.git eval/pdf-pii-bench
```

Then open `notebooks/Putting_It_Together.ipynb` with the `.venv` kernel, as before. (Everything here runs on a small CPU instance in a few seconds; we only read the results of the slower detectors.)

In [ ]:
import json
from collections import Counter
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
BENCH = ROOT / "eval" / "pdf-pii-bench" / "data"
texts = json.loads((BENCH / "text_data.json").read_text())
truth = json.loads((BENCH / "ground_truth.json").read_text())
print(len(texts), "documents,", sum(len(v) for v in truth.values()), "labeled spans")

"""
50 documents, 1121 labeled spans
"""

## $\color{teal}{\text{What does the data look like?}}$

Before any model, we look. Each document has its text and a category, and each labeled **span** is a piece of that text (a start, an end, and a label such as `private_person` or `account_number`) with a **subject**: whose information it is.

In [ ]:
print(Counter(d["category"] for d in texts.values()).most_common())
spans = [s for doc in truth.values() for s in doc]
print(Counter(s.get("subject", "person") for s in spans))
print(Counter(s["label"] for s in spans if s.get("subject", "person") == "person").most_common())

"""
[('tax_government', 9), ('financial_statement', 7), ('invoice', 7), ('email_footer', 6), ('legal_contract', 6), ('lease', 5), ('medical_intake', 5), ('resume', 5)]
Counter({'person': 877, 'org': 151, 'agency': 93})
[('private_date', 259), ('private_person', 229), ('account_number', 134), ('private_address', 96), ('private_phone', 63), ('private_email', 59), ('private_url', 25), ('secret', 12)]
"""

In [ ]:
lengths = sorted(len(d["text"]) for d in texts.values())
print("characters per document: min", lengths[0], "median", lengths[len(lengths) // 2], "max", lengths[-1])
lease = texts["lease_agreement"]["text"]
print(lease[:400])

"""
characters per document: min 1080 median 1765 max 37902
RESIDENTIAL LEASE AGREEMENT State of Texas · County of Travis This Residential Lease Agreement ("Agreement") is entered into this 12th day of May, 2025, by and between Rosalind Achebe ("Landlord") and Jonathan Pierce ("Tenant"). Landlord and Tenant may be referred to individually as a "Party" and collectively as the "Parties." 1. Premises. Landlord leases to Tenant, and Tenant leases from Landlord
"""

**Q. About a fifth of the labeled spans belong to an organization or an agency, and the benchmark doesn't score them at all. Why not?**

<details>
<summary>Click here for an answer</summary>


Because the product's job is to protect **people**. A company's toll-free number or a government agency's address is printed on purpose; removing it makes the document harder to read and protects no one. So, the benchmark leaves those spans neutral: a detector gets no credit for removing them and no penalty either.

</details>

## $\color{teal}{\text{How far does the simplest thing get?}}$

Our **baseline** is a list of regular expressions: patterns for emails, phone numbers, street addresses, dates, account numbers and web addresses. It's fast, it needs no model, and we can explain every finding. Here it is on the lease:

In [ ]:
from needtoknow.detectors import build_detector

regex = build_detector("regex")
for span in regex.detect(lease)[:12]:
    print(f"{span.label:16} {span.text}")

"""
private_address  2914 Bluebonnet Ln, Unit 4, Austin, TX 78704
private_date     June 1, 2025
private_date     May 31, 2026
account_number   445509812
private_phone    (512) 555-0180
private_address  2914 Bluebonnet Ln, Unit 4
private_email    rachebe.properties@gmail.com
private_phone    (512) 555-0121
private_phone    (512) 555-0180
private_address  2914 Bluebonnet Ln, Unit 4
private_date     05/12/2025
private_date     05/12/2025
"""

**Q. Before we measure it: what do you expect a list of patterns to miss?**

<details>
<summary>Click here for an answer</summary>


Names. There is no pattern for "a person's name": "Daniel Okafor" looks like any other pair of capitalized words. Patterns are good at things with a shape (an email has an `@`, a phone number has ten digits) and bad at things that are only known by meaning.

</details>

To measure it, we score the way a leak is felt: **character by character**. **Recall** is the share of the personal characters the detector removed, and **precision** is the share of what it removed that was personal. (These are the benchmark's own definitions; the repository's `evaluate.py` matches its scorer exactly.)

In [ ]:
from needtoknow.evaluate import evaluate

result = evaluate(ROOT / "eval" / "pdf-pii-bench", "regex", ROOT / "eval" / "mine")
print(f"recall {result['recall']:.3f}  precision {result['precision']:.3f}  seconds {result['seconds']}")
for label, r in result["recall_by_label"].items():
    print(f"  {label:18} {r:.3f}")

"""
recall 0.623  precision 0.930  seconds 0.1
  account_number     0.767
  private_address    0.814
  private_date       0.677
  private_email      1.000
  private_person     0.005
  private_phone      0.982
  private_url        0.916
  secret             0.029
"""

So, as expected: almost perfect on emails and phone numbers, and **0.005 recall on names**. The baseline already removes 62% of what it should, and when it removes something, it is right 93% of the time.

## $\color{teal}{\text{What do better models buy, and what do they cost?}}$

We tried two models that understand language: Presidio, Microsoft's open-source tool built on a spaCy model, and GLiNER, a small transformer that finds whatever kinds of entity you name. Each takes minutes to install and seconds to minutes to run, so the repository keeps their results in `eval/predictions`. We also tried the union of the baseline and GLiNER, and we list two published systems from the benchmark for comparison:

In [ ]:
published = {"privacy-filter (published)": (0.671, 0.963), "docscrubber (published)": (0.912, 0.774)}
print(f"{'detector':28} {'recall':>7} {'precision':>10} {'seconds':>8}")
for name in ["regex", "presidio", "gliner", "regex+gliner"]:
    s = json.loads((ROOT / "eval" / "predictions" / f"{name}.scores.json").read_text())
    print(f"{name:28} {s['recall']:7.3f} {s['precision']:10.3f} {s['seconds']:8.1f}")
for name, (r, p) in published.items():
    print(f"{name:28} {r:7.3f} {p:10.3f}")

"""
detector                      recall  precision  seconds
regex                          0.623      0.930      0.1
presidio                       0.702      0.460     12.1
gliner                         0.708      0.573    115.0
regex+gliner                   0.900      0.619    120.3
privacy-filter (published)     0.671      0.963
docscrubber (published)        0.912      0.774
"""

**Q. Which detector would you ship? Answer separately for the person sharing their lease, the person who reads the redacted copy, and the team running the service.**

<details>
<summary>Click here for an answer</summary>


There isn't one answer, and that's the point of naming the stakeholders in the proposal:

1. **The person sharing their lease** cares most about recall: one leaked account number is the failure they came to avoid. For them, `regex+gliner` at 0.90 is the clear choice.
2. **The person reading the redacted copy** cares about precision: a lease with "Tenant" and "Landlord" blacked out on every page is hard to read. Presidio, at 0.46, removes more that isn't personal than that is.
3. **The team running the service** cares about cost: the baseline answers in a hundredth of a second from a 555 MB image, while GLiNER needs two to five seconds per document, 2 GB of memory, and a 3.5 GB image.

</details>

## $\color{teal}{\text{Where does each one go wrong?}}$

A single number tells us how much is wrong, not what. **Error analysis** means reading the mistakes. First, what the baseline missed most often:

In [ ]:
for line in result["most_missed"][:10]:
    print(" ", line)

"""
  private_person: Jonathan Pierce
  private_person: Beatrice Kowalczyk
  private_person: Howard Ellison
  private_person: Rosalind Achebe
  private_person: Margaret R Thornberry
  private_person: Tessa Marie Holcomb
  private_date: 04/30
  private_person: Daniel Okafor
  private_person: Sarah Lindqvist
  private_person: Priya Raman
"""

Then, what each detector removed that it shouldn't have: its **false positives**, the spans that touch no labeled character at all.

In [ ]:
def false_positives(name):
    predictions = json.loads((ROOT / "eval" / "predictions" / f"{name}.json").read_text())
    wrong = Counter()
    for doc, spans in predictions.items():
        labeled = {i for g in truth[doc] for i in range(g["start"], g["end"])}
        for s in spans:
            if not labeled & set(range(s["start"], s["end"])):
                wrong[(s["label"], s["text"])] += 1
    return wrong.most_common(8)

for name in ["regex", "presidio"]:
    print(name)
    for (label, text), n in false_positives(name)[:6]:
        print(f"{n:6}  {label:16} {text}")

"""
regex
     6  account_number   1545-0029
     4  account_number   1545-0116
     3  account_number   2025-06
     2  account_number   482910
     2  private_date     February 1, 2027
     2  account_number   1545-0074
presidio
    56  private_address  U.S.
    35  private_person   Landlord
    23  private_person   Tenant
    23  private_date     2026
    21  private_address  Employee
    19  private_address  Premises
"""

**Q. Given these two lists, what would you try next, and how would you know it worked?**

<details>
<summary>Click here for an answer</summary>


1. **For the baseline's misses** (almost all names): add a model for names only and keep the patterns for everything with a shape. That is the `regex+gliner` union, and it is why its recall jumps to 0.90.
2. **For the false positives** ("U.S.", "Landlord", "Tenant", the baseline's IRS form numbers): a short list of words that are never personal in these documents, applied after any detector. GLiNER flags "Tenant" and "Landlord" too, so this helps the union as well.
3. **How we'd know**: score the changed predictions against the same benchmark and compare with the table above.

</details>

Let's try the second idea on the union's saved predictions: drop every finding whose text is on a short **stoplist**, then score again with the same function `evaluate` uses.

In [ ]:
from needtoknow.evaluate import score

STOP = {"U.S.", "the United States", "Landlord", "Tenant", "Employee", "Premises", "Client"}
union = json.loads((ROOT / "eval" / "predictions" / "regex+gliner.json").read_text())
kept = {doc: [s for s in spans if s["text"].strip() not in STOP] for doc, spans in union.items()}
for name, predictions in [("regex+gliner", union), ("with the stoplist", kept)]:
    s = score(texts, truth, predictions)
    print(f"{name:18} recall {s['recall']:.3f}  precision {s['precision']:.3f}")

"""
regex+gliner       recall 0.900  precision 0.619
with the stoplist  recall 0.900  precision 0.644
"""

**Q. Precision went up and recall didn't move. Should we ship the stoplist?**

<details>
<summary>Click here for an answer</summary>


Probably, but not on this evidence alone. We built the list by reading this benchmark's mistakes, so of course it helps on this benchmark. The honest test is documents we haven't looked at: new synthetic ones, or the ones your testers upload in Homework 3. And a list can hurt there, for instance if someone's last name is Client. So, a change chosen from the errors is measured again on data it wasn't chosen from.

</details>

## $\color{teal}{\text{Does the skeleton do it end to end?}}$

A benchmark score is not a product. The skeleton is: one page to upload a PDF, a list of findings to review, and a redacted copy to download, deployed on Jetstream2 behind one address. Its interface is two endpoints, so we can call it from Python exactly as the page does. (When I'm testing this, I open the address in my browser first; if the instance is shelved, the cell below says so.)

In [ ]:
import httpx
import pymupdf

URL = "http://149.165.170.89:8000"
doc = pymupdf.open()
page = doc.new_page()
for i, line in enumerate(["LEASE AGREEMENT",
                          "Tenant: Maria Delgado, phone (202) 555-0143, email maria.delgado@example.com",
                          "Premises: 1450 Oak Street, Apt 3B, Arlington, VA 22201"]):
    page.insert_text((50, 72 + 18 * i), line, fontsize=10)
pdf = doc.tobytes()

try:
    found = httpx.post(f"{URL}/api/detect", files={"file": ("lease.pdf", pdf, "application/pdf")}, timeout=httpx.Timeout(60, connect=5)).json()
    print(found["detector"], "found", len(found["findings"]), "items in", found["seconds"], "seconds")
    for f in found["findings"]:
        print(f"  page {f['page']}  {f['label']:16} {f['text']}")
    remove = json.dumps([{"page": f["page"], "text": f["text"]} for f in found["findings"]])
    redacted = httpx.post(f"{URL}/api/redact", files={"file": ("lease.pdf", pdf, "application/pdf")},
                          data={"remove": remove}, timeout=httpx.Timeout(60, connect=5))
    left = pymupdf.open(stream=redacted.content, filetype="pdf")[0].get_text()
    print("still in the redacted copy:", [f["text"] for f in found["findings"] if f["text"] in left])
except httpx.TransportError:
    print("The demo instance isn't reachable right now; start the app in a VS Code terminal with `uv run uvicorn needtoknow.api.main:app` and set URL to http://localhost:8000.")

"""
The demo instance isn't reachable right now; start the app in a VS Code terminal with `uv run uvicorn needtoknow.api.main:app` and set URL to http://localhost:8000.
"""

**Q. The deployed skeleton finds the phone number, the email and the address, but not "Maria Delgado". Is it broken?**

<details>
<summary>Click here for an answer</summary>


No, it's doing exactly what we measured: it runs the baseline, and the baseline's recall on names is 0.005. The skeleton was deployed with the cheapest detector on purpose, so that the whole path (upload, detect, review, redact, download) worked on day one.

Switching to the union is one setting and one rebuild, because the detector is chosen by configuration:

```bash
echo "NTK_DETECTOR=regex+gliner" >> .env
DETECTOR_EXTRA=gliner docker compose up -d --build
```

</details>

## $\color{teal}{\text{Take-home message}}$

In this notebook, we looked at the data before any model, measured a baseline, compared better detectors against what each stakeholder cares about, read the mistakes to choose the next change, and called the deployed skeleton from Python. Your project's Deliverable 2 asks for the same path: a skeleton that works end to end, a baseline you can measure, and a clear next step.

**You've taken a project from a question to a running, measured skeleton!**